# Deep Agents
[deepagents](https://docs.langchain.com/oss/python/deepagents/overview) is a standalone library built on top of LangChain’s core building blocks for agents. It uses the LangGraph runtime for durable execution, streaming, human-in-the-loop, and other features.

| Capability                | Our agentic notebook (it_incident_agentic)                                   | Deep Agents (same loop, more built-ins)                |
|---------------------------|---------------------------------------------------------|--------------------------------------------------------|
| Agent loop                | `create_react_agent`                                      | `create_deep_agent`                                      |
| Subagent delegation       | Manual tool calls                                       | Built-in task() tool spawns isolated subagents         |
| Context management        | Messages list grows unbounded                           | File system tools offload large outputs                |
| Task planning             | Implicit in system prompt                               | Built-in write_todos tracks progress explicitly        |
| Memory across sessions    | None                                                    | Built-in long-term memory store (file system)                        |

### Libraries

In [0]:
%pip install langgraph langchain deepagents databricks-langchain -q 

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

### LLM

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",  # no endpoint setup
    temperature=0,
)

### Agent Tools

In [0]:
from langchain_core.tools import tool

@tool
def collect_alert(raw_alert: str) -> str:
    """Parse and normalise a raw IT monitoring alert."""
    response = llm.invoke(
        f"Parse this alert and extract: source system, timestamp, "
        f"affected service, error type, impact scope.\n\nAlert: {raw_alert}"
    )
    return response.content


@tool
def screen_alert(alert: str) -> str:
    """Assign severity (LOW/MEDIUM/HIGH/CRITICAL) and urgency to an alert."""
    response = llm.invoke(
        f"Reply in JSON only: {{\"severity\": \"LOW|MEDIUM|HIGH|CRITICAL\", "
        f"\"urgency\": \"ROUTINE|URGENT|IMMEDIATE\"}}.\n\nAlert: {alert}"
    )
    return response.content


@tool
def analyze_alert(alert: str, severity: str) -> str:
    """Identify the root cause. Only call this for HIGH or CRITICAL severity."""
    response = llm.invoke(
        f"Identify the most likely root cause.\n\nAlert: {alert}\nSeverity: {severity}"
    )
    return response.content


@tool
def fix_incident(root_cause: str, severity: str) -> str:
    """Generate step-by-step remediation steps."""
    response = llm.invoke(
        f"Provide numbered remediation steps.\n\nRoot cause: {root_cause}\nSeverity: {severity}"
    )
    return response.content


@tool
def escalate(alert: str, reason: str) -> str:
    """Escalate to a human engineer when the agent cannot resolve the incident."""
    # In production: call PagerDuty / OpsGenie API here
    return f"ESCALATED: {reason} | Alert: {alert[:100]}"

tools = [collect_alert, screen_alert, analyze_alert, fix_incident, escalate]

### Sub-Agents

In [0]:

collector_subagent = {
    "name": "collector",
    "description": "Parses and normalises raw IT alerts. Call this first with the raw alert text.",
    "tools": [collect_alert],
    "system_prompt": "You are an IT alert ingestion agent. Extract: source system, timestamp, affected service, error type, impact scope.",
}

screener_subagent = {
    "name": "screener",
    "description": "Assigns severity (LOW/MEDIUM/HIGH/CRITICAL) and urgency. Call after collector.",
    "tools": [screen_alert],
    "system_prompt": "Classify alerts. Reply in JSON: {\"severity\": \"...\", \"urgency\": \"...\"}",
}

analyzer_subagent = {
    "name": "analyzer",
    "description": "Identifies root cause. Only call for HIGH or CRITICAL severity.",
    "tools": [analyze_alert],
    "system_prompt": "You are a root cause analysis agent. Be concise and specific.",
}

fixer_subagent = {
    "name": "fixer",
    "description": "Generates step-by-step remediation actions given a root cause and severity.",
    "tools": [fix_incident, escalate],
    "system_prompt": "Provide numbered, actionable remediation steps.",
}


### Deep Agent (Orchestrator)

In [0]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

app = create_deep_agent(
    model = llm,
    subagents=[collector_subagent, screener_subagent, analyzer_subagent, fixer_subagent],
    system_prompt="""You are an autonomous IT incident management agent.
                    When given an alert:
                    1. Use collector to normalise it
                    2. Use screener to determine severity and urgency
                    3. If HIGH or CRITICAL → use analyzer, then fixer
                    4. If LOW or MEDIUM → use fixer directly
                    5. If root cause is unclear after analysis → escalate
                    Never fabricate information.""",
    backend=FilesystemBackend(root_dir=".", virtual_mode=True)
)

### Process Alerts

In [0]:
from langchain_core.messages import HumanMessage

def process_pending_alerts():
    from pyspark.sql.functions import col

    pending = spark.table("main.it_ops.alert_queue") \
                   .filter(col("processed") == False)
    rows = pending.collect()

    if not rows:
        print("No pending alerts.")
        return

    for row in rows:
        print(f"\nProcessing: {row.raw_alert[:60]}...")

        result = app.invoke({
            "messages": [HumanMessage(content=f"Handle this IT alert: {row.raw_alert}")]
        })

        # The agent's final message contains its reasoning + conclusion
        final = result["messages"][-1].content
        print(f"Agent conclusion:\n{final}")

        # Log the full reasoning trace to Delta
        trace = "\n".join([m.content for m in result["messages"] if hasattr(m, "content")])
        spark.createDataFrame(
            [(row.raw_alert[:500], final[:1000], trace[:5000])],
            ["alert", "conclusion", "trace"]
        ).write.mode("append").option("mergeSchema", "true").saveAsTable("main.it_ops.incident_log")

        spark.sql(f"""
            UPDATE main.it_ops.alert_queue SET processed = TRUE
            WHERE raw_alert = '{row.raw_alert.replace("'", "''")}'
        """)



In [0]:
from datetime import datetime

new_alert = spark.createDataFrame([
    ("CPU spike to 100% on prod-db-02 at 00:32 UTC. 500 errors on API gateway.", datetime.utcnow(), False),
    ("Disk usage at 99% on prod-storage-03. Write latency > 900ms.",             datetime.utcnow(), False),
], ["raw_alert", "created_at", "processed"])

new_alert.write.mode("append").saveAsTable("main.it_ops.alert_queue")
print("Alerts queued. Run process.")

/home/spark-48789ff1-491a-4ddf-a01a-5c/.ipykernel/4496/command-6445004805804624-2806890000:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ("CPU spike to 100% on prod-db-02 at 00:32 UTC. 500 errors on API gateway.", datetime.utcnow(), False),
/home/spark-48789ff1-491a-4ddf-a01a-5c/.ipykernel/4496/command-6445004805804624-2806890000:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ("Disk usage at 99% on prod-storage-03. Write latency > 900ms.",             datetime.utcnow(), False),


Alerts queued. Run process.


In [0]:
process_pending_alerts()


Processing: CPU spike to 100% on prod-db-02 at 00:32 UTC. 500 errors on ...
Agent conclusion:
The IT alert regarding the CPU spike to 100% on prod-db-02 and 500 errors on the API gateway has been thoroughly addressed. The root cause has been identified as potentially related to database overload or query issues, API gateway configuration or code issues, resource constraints, or external attacks or traffic spikes. 

Remediation steps have been generated, including assessing and identifying the root cause, implementing emergency optimizations, adjusting configurations, scaling up resources, activating protection services, monitoring performance, applying code fixes, conducting post-incident reviews, notifying stakeholders, and reviewing the incident response plan. 

It is essential to follow these steps in a timely and coordinated manner, considering the urgency and severity of the issue. Continuous monitoring and analysis will help in quickly identifying any new issues and ensuring the